In [121]:
# 04_model_implementation.ipynb
# 100% Artifact + Paper compliant (IMPLEMENTATION ONLY)
# ============================================================
"""
This notebook:
- Loads preprocessing outputs (cleaned, scaled, split, and feature lists).
- Trains all paper models (FFT-MLP, FFT-RF, CNN-BiLSTM, and fusion models).
- Saves trained models and inference outputs for the evaluation notebook.

Outputs:
- Trained models (saved to `models/`).
- Inference outputs (saved to `outputs/`).
- Metadata (saved to `metadata/`).

Note: Final metrics and plots are computed in the evaluation notebook.
"""

'\nThis notebook:\n- Loads preprocessing outputs (cleaned, scaled, split, and feature lists).\n- Trains all paper models (FFT-MLP, FFT-RF, CNN-BiLSTM, and fusion models).\n- Saves trained models and inference outputs for the evaluation notebook.\n\nOutputs:\n- Trained models (saved to `models/`).\n- Inference outputs (saved to `outputs/`).\n- Metadata (saved to `metadata/`).\n\nNote: Final metrics and plots are computed in the evaluation notebook.\n'

In [122]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder


In [123]:
# Reproducibility and Device Setup
# ----------------------------
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Using device: {DEVICE}")


 Using device: cpu


In [124]:
# Paths and Directories
# ----------------------------
DATA_DIR = Path("../data/processed/cmi_sensor_data")
OUT_DIR = Path("../models_artifacts")
MODELS_DIR = OUT_DIR / "models"
OUTPUTS_DIR = OUT_DIR / "outputs"
META_DIR = OUT_DIR / "metadata"

# Create output directories if they don't exist
for directory in [MODELS_DIR, OUTPUTS_DIR, META_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Input paths (preprocessing outputs)
TRAIN_CLEAN = DATA_DIR / "train_clean.csv"
TEST_CLEAN = DATA_DIR / "test_clean.csv"
FEATURES_JSON = DATA_DIR / "feature_cols.json"
SPLIT_JSON = DATA_DIR / "split_sequence_ids.json"
# Verify all required files exist
for path in [TRAIN_CLEAN, TEST_CLEAN, FEATURES_JSON, SPLIT_JSON]:
    if not path.exists():
        raise FileNotFoundError(f" Missing required file: {path}")


In [125]:
# Copy metadata for evaluation notebook
(META_DIR / "feature_cols.json").write_text(FEATURES_JSON.read_text(), encoding="utf-8")
(META_DIR / "split_sequence_ids.json").write_text(SPLIT_JSON.read_text(), encoding="utf-8")
# Load Data
print(" Loading data...")
train_df = pd.read_csv(TRAIN_CLEAN)
test_df = pd.read_csv(TEST_CLEAN)

 Loading data...


In [126]:
# Load feature columns and split IDs
with open(FEATURES_JSON, "r", encoding="utf-8") as file:
    FEATURE_COLS = json.load(file)["feature_cols"]

with open(SPLIT_JSON, "r", encoding="utf-8") as file:
    split_data = json.load(file)


In [127]:
# Column names
SEQ_COL = "sequence_id"
GESTURE_COL = "gesture"
BEHAVIOR_COL = "behavior"

# Extract train/val sequence IDs
train_seq_ids = np.array(split_data["train_seq_ids"])
val_seq_ids = np.array(split_data["val_seq_ids"])
BFRB_GESTURES = {
    "Neck - scratch",
    "Eyebrow - pull hair",
    "Forehead - scratch",
    "Forehead - pull hairline",
    "Above ear - pull hair",
    "Neck - pinch skin",
    "Eyelash - pull hair",
    "Cheek - pinch skin",
}

In [128]:
# Verify required columns
for column in [SEQ_COL, GESTURE_COL, BEHAVIOR_COL]:
    if column not in train_df.columns:
        raise KeyError(f" Missing column '{column}' in train_clean.csv")


In [129]:
# Feature Groups (Paper-Aligned)
IMU_COLS = [col for col in FEATURE_COLS if col.startswith(("acc_", "rot_"))]
THM_COLS = [col for col in FEATURE_COLS if col.startswith("thm_")]
TOF_COLS = [col for col in FEATURE_COLS if col.startswith("tof_")]


In [130]:
# Sanity checks
assert len(FEATURE_COLS) == 332, f" Expected 332 sensor features, found {len(FEATURE_COLS)}"
assert len(TOF_COLS) == 320, f" Expected 320 TOF features, found {len(TOF_COLS)}"

IMU_THM_COLS = IMU_COLS + THM_COLS


In [131]:
# Label Builders (Sequence-Level)
def make_binary_y_from_firstrow(df_firstrow: pd.DataFrame) -> np.ndarray:
    """
    Create binary labels (BFRB=1, non-BFRB=0) from the first row of each sequence.

    Args:
        df_firstrow: DataFrame with one row per sequence.

    Returns:
        Binary labels as a numpy array.
    """
    gestures = df_firstrow[GESTURE_COL].astype(str)
    return gestures.isin(BFRB_GESTURES).astype(int).to_numpy()


def make_macro_y_from_firstrow(df_firstrow: pd.DataFrame) -> np.ndarray:
    """
    Create macro labels (gesture or 'non_target') from the first row of each sequence.

    Args:
        df_firstrow: DataFrame with one row per sequence.

    Returns:
        Macro labels as a numpy array.
    """
    gestures = df_firstrow[GESTURE_COL].astype(str).to_numpy()
    is_bfrb = np.isin(gestures, list(BFRB_GESTURES))
    return np.where(is_bfrb, gestures, "non_target").astype(object)

def sequence_firstrow(df: pd.DataFrame) -> pd.DataFrame:
    """
    Extract the first row of each sequence.

    Args:
        df: Input DataFrame.

    Returns:
        DataFrame with one row per sequence.
    """
    return df.sort_values([SEQ_COL]).groupby(SEQ_COL, as_index=False).first()


In [132]:
# ==============================
# PAPER-COMPLIANT TRAINING LABELS (18 gesture classes)
# ==============================

from sklearn.preprocessing import LabelEncoder
import json
from pathlib import Path

# Build sequence-level rows
train_first = sequence_firstrow(
    train_df[train_df[SEQ_COL].isin(train_seq_ids)].copy()
)

val_first = sequence_firstrow(
    train_df[train_df[SEQ_COL].isin(val_seq_ids)].copy()
)

# Paper requires raw gesture as output (18 classes)
gesture_classes = sorted(train_first[GESTURE_COL].astype(str).unique().tolist())
print("NUM_CLASSES:", len(gesture_classes))
assert len(gesture_classes) == 18, "Paper requires 18 gesture classes."

# Create label encoder from gesture
le_gesture = LabelEncoder()
le_gesture.fit(gesture_classes)

y_train_gesture_idx = le_gesture.transform(
    train_first[GESTURE_COL].astype(str)
)

y_val_gesture_idx = le_gesture.transform(
    val_first[GESTURE_COL].astype(str)
)

# Save class mapping for evaluation notebook
META_DIR = Path("../models_artifacts/metadata")
META_DIR.mkdir(parents=True, exist_ok=True)

class_mapping = {
    "classes": gesture_classes,
    "class_to_idx": {c: i for i, c in enumerate(gesture_classes)}
}

with open(META_DIR / "class_mapping.json", "w", encoding="utf-8") as f:
    json.dump(class_mapping, f, indent=2)

NUM_CLASSES = len(gesture_classes)


NUM_CLASSES: 18


In [133]:
# Extract first rows for train/val sequences
train_first = sequence_firstrow(train_df[train_df[SEQ_COL].isin(train_seq_ids)].copy())
val_first = sequence_firstrow(train_df[train_df[SEQ_COL].isin(val_seq_ids)].copy())


In [ ]:
# Save validation sequence IDs and labels
np.save(OUTPUTS_DIR / "val_seq_ids.npy", val_first[SEQ_COL].to_numpy())
y_val_bin = make_binary_y_from_firstrow(val_first)
y_val_macro = make_macro_y_from_firstrow(val_first)
# Debugging points
from collections import Counter
print("DEBUG y_val_bin:", Counter(y_val_bin.tolist()))
print("DEBUG y_val_macro unique:", len(set(map(str, y_val_macro))))
print("DEBUG y_val_macro top 10:", Counter(map(str, y_val_macro)).most_common(10))

np.save(OUTPUTS_DIR / "y_val_bin.npy", y_val_bin)
np.save(OUTPUTS_DIR / "y_val_macro.npy", y_val_macro)
# Debugging
y_check = np.load(OUTPUTS_DIR / "y_val_bin.npy")
m_check = np.load(OUTPUTS_DIR / "y_val_macro.npy", allow_pickle=True)

from collections import Counter
print("POST-SAVE y_val_bin:", Counter(y_check.tolist()))
print("POST-SAVE y_val_macro unique:", len(set(map(str, m_check))))


DEBUG y_val_bin: Counter({1: 953, 0: 565})
DEBUG y_val_macro unique: 9
DEBUG y_val_macro top 10: [('non_target', 565), ('Neck - scratch', 132), ('Eyelash - pull hair', 123), ('Above ear - pull hair', 122), ('Forehead - scratch', 122), ('Forehead - pull hairline', 116), ('Neck - pinch skin', 114), ('Eyebrow - pull hair', 113), ('Cheek - pinch skin', 111)]
POST-SAVE y_val_bin: Counter({1: 953, 0: 565})
POST-SAVE y_val_macro unique: 9


In [135]:
# Gesture Label Encoding (Paper-Compliant)
"""
This section:
- Encodes gesture labels for 18-class classification (paper requirement).
- Saves class mappings for reproducibility.
- Uses sequence-level labels (first row of each sequence).
"""

from sklearn.preprocessing import LabelEncoder

# Initialize label encoder
gesture_label_encoder = LabelEncoder()

# Extract all unique gesture classes from the training data
gesture_classes = sorted(train_df[GESTURE_COL].astype(str).unique().tolist())
print(f"🔢 Number of gesture classes: {len(gesture_classes)}")

# Paper requires 18 gesture classes
assert len(gesture_classes) == 18, (
    f" Expected 18 gesture classes (paper requirement), "
    f"found {len(gesture_classes)}. Check data or preprocessing."
)

# Fit the label encoder on all gesture classes
gesture_label_encoder.fit(gesture_classes)

# Create mappings for reference
classes = gesture_label_encoder.classes_.tolist()
class_to_index = {cls: idx for idx, cls in enumerate(classes)}
index_to_class = {idx: cls for cls, idx in class_to_index.items()}

# Save class mappings for reproducibility
with open(META_DIR / "class_mapping.json", "w", encoding="utf-8") as file:
    json.dump(
        {
            "classes": classes,
            "class_to_idx": class_to_index,
            "note": "18-class gesture labels (paper-compliant)"
        },
        file,
        indent=2
    )
print(f" Class mappings saved to: {META_DIR / 'class_mapping.json'}")

# Encode sequence-level gesture labels for train/val sets
# (using first row of each sequence, as per paper methodology)
y_train_gesture_idx = gesture_label_encoder.transform(train_first[GESTURE_COL].astype(str))
y_val_gesture_idx = gesture_label_encoder.transform(val_first[GESTURE_COL].astype(str))

print(
    f" Train gesture labels shape: {y_train_gesture_idx.shape} "
    f"(indices: {y_train_gesture_idx.min()} to {y_train_gesture_idx.max()})"
)
print(
    f" Validation gesture labels shape: {y_val_gesture_idx.shape} "
    f"(indices: {y_val_gesture_idx.min()} to {y_val_gesture_idx.max()})"
)


🔢 Number of gesture classes: 18
 Class mappings saved to: ../models_artifacts/metadata/class_mapping.json
 Train gesture labels shape: (6070,) (indices: 0 to 17)
 Validation gesture labels shape: (1518,) (indices: 0 to 17)


In [136]:
# FFT Feature Extraction
def choose_fft_length(df: pd.DataFrame, seq_col: str, min_len: int = 35) -> tuple[int, int]:
    """
    Choose FFT length (L) and frequency bins (K) deterministically.

    Args:
        df: Input DataFrame.
        seq_col: Column name for sequence IDs.
        min_len: Minimum sequence length.

    Returns:
        Tuple of (L, K).
    """
    sequence_lengths = df.groupby(seq_col).size()
    L = int(round(float(sequence_lengths.mean())))
    L = max(L, min_len)
    K = min(64, (L // 2) + 1)
    return L, K

def extract_fft_features_per_sequence(
    df: pd.DataFrame,
    seq_col: str,
    feature_cols: list[str],
    L: int,
    K: int
) -> tuple[np.ndarray, np.ndarray]:
    """
    Extract FFT features for each sequence.

    Args:
        df: Input DataFrame.
        seq_col: Column name for sequence IDs.
        feature_cols: List of feature columns.
        L: Sequence length.
        K: Number of frequency bins.

    Returns:
        Tuple of (FFT features, sequence IDs).
    """
    sequence_ids = df[seq_col].drop_duplicates().to_numpy()
    num_sequences = len(sequence_ids)
    num_features = len(feature_cols)
    fft_features = np.zeros((num_sequences, num_features * K), dtype=np.float32)

    for i, seq_id in enumerate(sequence_ids):
        sequence_data = df[df[seq_col] == seq_id][feature_cols].to_numpy(dtype=np.float32)
        T = sequence_data.shape[0]
        if T >= L:
            padded_data = sequence_data[:L, :]
        else:
            pad = np.zeros((L - T, sequence_data.shape[1]), dtype=np.float32)
            padded_data = np.vstack([sequence_data, pad])

        fft = np.fft.rfft(padded_data, axis=0)
        magnitude = np.abs(fft)[:K, :]
        fft_features[i, :] = magnitude.reshape(-1)

    return fft_features, sequence_ids


In [137]:
# Determine FFT parameters
L_fft, K_fft = choose_fft_length(train_df[train_df[SEQ_COL].isin(train_seq_ids)], SEQ_COL)
with open(META_DIR / "fft_config.json", "w", encoding="utf-8") as file:
    json.dump({"L_fft": L_fft, "K_fft": K_fft, "K_cap": 64, "seed": RANDOM_SEED}, file, indent=2)

In [138]:
# Extract FFT features (aligned with train_first/val_first)
def extract_fft_features_aligned(rows_df: pd.DataFrame, first_df: pd.DataFrame, cols: list[str], L: int, K: int) -> np.ndarray:
    """
    Extract FFT features aligned with the order in first_df.

    Args:
        rows_df: DataFrame with all rows.
        first_df: DataFrame with first rows (for order).
        cols: List of feature columns.
        L: Sequence length.
        K: Number of frequency bins.

    Returns:
        Aligned FFT features.
    """
    fft_features, sequence_ids = extract_fft_features_per_sequence(rows_df, SEQ_COL, cols, L, K)
    order = first_df[SEQ_COL].to_numpy()
    sequence_id_to_index = {seq_id: idx for idx, seq_id in enumerate(sequence_ids)}
    return np.stack([fft_features[sequence_id_to_index[seq_id]] for seq_id in order], axis=0)


In [139]:
# Extract FFT features for train/val sets
train_rows = train_df[train_df[SEQ_COL].isin(train_seq_ids)].copy()
val_rows = train_df[train_df[SEQ_COL].isin(val_seq_ids)].copy()

X_train_fft_all = extract_fft_features_aligned(train_rows, train_first, FEATURE_COLS, L_fft, K_fft)
X_val_fft_all = extract_fft_features_aligned(val_rows, val_first, FEATURE_COLS, L_fft, K_fft)
X_train_fft_imu_thm = extract_fft_features_aligned(train_rows, train_first, IMU_THM_COLS, L_fft, K_fft)
X_val_fft_imu_thm = extract_fft_features_aligned(val_rows, val_first, IMU_THM_COLS, L_fft, K_fft)

In [140]:
# Model 1: FFT-MLP (All Sensors)
# ----------------------------
print(" Training FFT-MLP (All Sensors)...")
mlp_all = MLPClassifier(
    hidden_layer_sizes=(128, 64, 64),
    batch_size=256,
    max_iter=30,
    random_state=RANDOM_SEED,
    learning_rate_init=0.001,
)
mlp_all.fit(X_train_fft_all, y_train_gesture_idx)
# Save model and logits
joblib.dump(
    {
        "model": mlp_all,
        "label_encoder_classes": classes,
        "fft_cols": "ALL",
        "L_fft": L_fft,
        "K_fft": K_fft
    },
    MODELS_DIR / "fft_mlp_all.joblib"
)

val_proba = mlp_all.predict_proba(X_val_fft_all)
val_logits = np.log(np.clip(val_proba, 1e-12, 1.0))
np.save(OUTPUTS_DIR / "logits_val_fft_mlp_all.npy", val_logits)


 Training FFT-MLP (All Sensors)...


/Users/ujjwaldahiya/CAP-@-EXP/2026-winter-capstone-project-2026winter-capstone-group-5/venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(


In [141]:
# Model 2: FFT-MLP (IMU + THM)
print(" Training FFT-MLP (IMU + THM)...")
mlp_imu_thm = MLPClassifier(
    hidden_layer_sizes=(128, 64, 64),
    batch_size=256,
    max_iter=30,
    random_state=RANDOM_SEED,
    learning_rate_init=0.001,
)
mlp_imu_thm.fit(X_train_fft_imu_thm, y_train_gesture_idx)

joblib.dump(
    {
        "model": mlp_imu_thm,
        "label_encoder_classes": classes,
        "fft_cols": "IMU+THM",
        "L_fft": L_fft,
        "K_fft": K_fft
    },
    MODELS_DIR / "fft_mlp_imu_thm.joblib"
)

val_proba = mlp_imu_thm.predict_proba(X_val_fft_imu_thm)
val_logits = np.log(np.clip(val_proba, 1e-12, 1.0))
np.save(OUTPUTS_DIR / "logits_val_fft_mlp_imu_thm.npy", val_logits)


 Training FFT-MLP (IMU + THM)...


/Users/ujjwaldahiya/CAP-@-EXP/2026-winter-capstone-project-2026winter-capstone-group-5/venv/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(


In [142]:
# Model 3: FFT-RandomForest (All Sensors)
print(" Training FFT-RandomForest (All Sensors)...")
rf_all = RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1)
rf_all.fit(X_train_fft_all, y_train_gesture_idx)

joblib.dump(
    {
        "model": rf_all,
        "label_encoder_classes": classes,
        "fft_cols": "ALL",
        "L_fft": L_fft,
        "K_fft": K_fft
    },
    MODELS_DIR / "fft_rf_all.joblib"
)

val_proba = rf_all.predict_proba(X_val_fft_all)
val_logits = np.log(np.clip(val_proba, 1e-12, 1.0))
np.save(OUTPUTS_DIR / "logits_val_fft_rf_all.npy", val_logits)


 Training FFT-RandomForest (All Sensors)...


In [143]:
# Torch Utilities for TOF Model
def tof_to_5x8x8(df_sequence: pd.DataFrame) -> np.ndarray:
    """
    Reshape TOF features to (T, 5, 8, 8).

    Args:
        df_sequence: DataFrame for a single sequence.

    Returns:
        Reshaped TOF features.
    """
    tof_features = df_sequence[TOF_COLS].to_numpy(dtype=np.float32)
    return tof_features.reshape(tof_features.shape[0], 5, 8, 8)

def pad_or_truncate_time(array: np.ndarray, L: int) -> np.ndarray:
    """
    Pad or truncate time dimension to length L.

    Args:
        array: Input array.
        L: Target length.

    Returns:
        Padded or truncated array.
    """
    T = array.shape[0]
    if T >= L:
        return array[:L]
    pad = np.zeros((L - T, *array.shape[1:]), dtype=array.dtype)
    return np.concatenate([array, pad], axis=0)


In [144]:
# Fixed time length for torch models
L_TORCH = int(round(float(train_rows.groupby(SEQ_COL).size().mean())))
L_TORCH = max(L_TORCH, 35)

In [145]:
class TOFSequenceDataset(Dataset):
    """PyTorch Dataset for TOF sequences."""
    def __init__(self, rows_df: pd.DataFrame, first_df: pd.DataFrame, y_idx: np.ndarray):
        self.rows_df = rows_df
        self.sequence_ids = first_df[SEQ_COL].to_numpy()
        self.labels = y_idx

    def __len__(self) -> int:
        return len(self.sequence_ids)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        sequence_id = self.sequence_ids[index]
        sequence_data = self.rows_df[self.rows_df[SEQ_COL] == sequence_id]
        tof_features = tof_to_5x8x8(sequence_data)
        padded_tof = pad_or_truncate_time(tof_features, L_TORCH)
        return torch.from_numpy(padded_tof), torch.tensor(self.labels[index], dtype=torch.long)


In [146]:
class CNNBiLSTM_TOF(nn.Module):
    """CNN-BiLSTM model for TOF features."""
    def __init__(self, num_classes: int):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(5, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.projection = nn.Linear(64, 128)
        self.bilstm = nn.LSTM(input_size=128, hidden_size=128, batch_first=True, bidirectional=True)
        self.head = nn.Linear(256, num_classes)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        B, L, C, H, W = x.shape
        x_reshaped = x.view(B * L, C, H, W)
        cnn_features = self.cnn(x_reshaped).view(B * L, 64)
        embedded = self.projection(cnn_features).view(B, L, 128)
        lstm_out, _ = self.bilstm(embedded)
        last_hidden = lstm_out[:, -1, :]
        logits = self.head(last_hidden)
        return logits, last_hidden


In [147]:
def train_torch_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int = 5,
    learning_rate: float = 1e-3,
    weight_decay: float = 1e-4
) -> nn.Module:
    """
    Train a PyTorch model.

    Args:
        model: Model to train.
        train_loader: Training DataLoader.
        val_loader: Validation DataLoader.
        epochs: Number of epochs.
        learning_rate: Learning rate.
        weight_decay: Weight decay.

    Returns:
        Trained model.
    """
    model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    loss_function = nn.CrossEntropyLoss()

    for epoch in range(1, epochs + 1):
        model.train()
        for batch_features, batch_labels in train_loader:
            batch_features, batch_labels = batch_features.to(DEVICE), batch_labels.to(DEVICE)
            optimizer.zero_grad()
            logits, _ = model(batch_features)
            loss = loss_function(logits, batch_labels)
            loss.backward()
            optimizer.step()

        # Validation sanity check
        model.eval()
        with torch.no_grad():
            val_features, val_labels = next(iter(val_loader))
            val_features = val_features.to(DEVICE)
            logits, _ = model(val_features)
            print(f"📊 Epoch {epoch}: Val batch logits shape = {tuple(logits.shape)}")

    return model

In [148]:
# Model 4: CNN-BiLSTM (TOF)
print(" Training CNN-BiLSTM (TOF)...")
train_dataset_tof = TOFSequenceDataset(train_rows, train_first, y_train_gesture_idx)
val_dataset_tof   = TOFSequenceDataset(val_rows,   val_first,   y_val_gesture_idx)


train_loader_tof = DataLoader(train_dataset_tof, batch_size=32, shuffle=True)
val_loader_tof = DataLoader(val_dataset_tof, batch_size=32, shuffle=False)

NUM_CLASSES = len(classes)
cnn_bilstm = CNNBiLSTM_TOF(NUM_CLASSES)
cnn_bilstm = train_torch_model(cnn_bilstm, train_loader_tof, val_loader_tof, epochs=5)

# Save model and logits
torch.save(
    {
        "state_dict": cnn_bilstm.state_dict(),
        "num_classes": NUM_CLASSES,
        "classes": classes,
        "L_TORCH": L_TORCH,
        "seed": RANDOM_SEED
    },
    MODELS_DIR / "cnn_bilstm_tof.pt"
)

cnn_bilstm.eval()
val_logits_list = []
with torch.no_grad():
    for val_features, _ in val_loader_tof:
        val_features = val_features.to(DEVICE)
        logits, _ = cnn_bilstm(val_features)
        val_logits_list.append(logits.cpu().numpy())
val_logits_tof = np.vstack(val_logits_list)
np.save(OUTPUTS_DIR / "logits_val_cnn_bilstm_tof.npy", val_logits_tof)


 Training CNN-BiLSTM (TOF)...
📊 Epoch 1: Val batch logits shape = (32, 18)
📊 Epoch 2: Val batch logits shape = (32, 18)
📊 Epoch 3: Val batch logits shape = (32, 18)
📊 Epoch 4: Val batch logits shape = (32, 18)
📊 Epoch 5: Val batch logits shape = (32, 18)


In [149]:
# Model 5: Late Fusion
print(" Computing Late Fusion logits...")
val_logits_mlp_imu_thm = np.load(OUTPUTS_DIR / "logits_val_fft_mlp_imu_thm.npy")
val_logits_tof = np.load(OUTPUTS_DIR / "logits_val_cnn_bilstm_tof.npy")

# Paper weights: 0.3 * FFT-MLP(IMU+THM) + 0.7 * CNN-BiLSTM(TOF)
fusion_logits = 0.3 * val_logits_mlp_imu_thm + 0.7 * val_logits_tof
np.save(OUTPUTS_DIR / "logits_val_late_fusion.npy", fusion_logits)


 Computing Late Fusion logits...


In [150]:
# Model 6: Intermediate Fusion
class FFTMLPBranch(nn.Module):
    """FFT-MLP branch for intermediate fusion."""
    def __init__(self, input_dim: int):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU()
        )
        self.embedding = nn.Linear(64, 128)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        hidden = self.network(x)
        return torch.relu(self.embedding(hidden))

class IntermediateFusionModel(nn.Module):
    """Intermediate fusion model."""
    def __init__(self, fft_input_dim: int, num_classes: int):
        super().__init__()
        self.fft_branch = FFTMLPBranch(fft_input_dim)
        self.tof_branch = CNNBiLSTM_TOF(num_classes)
        self.tof_projection = nn.Linear(256, 128)
        self.head = nn.Linear(256, num_classes)

    def forward(self, fft_features: torch.Tensor, tof_features: torch.Tensor) -> torch.Tensor:
        fft_embedding = self.fft_branch(fft_features)
        tof_logits, tof_embedding_256 = self.tof_branch(tof_features)
        tof_embedding = torch.relu(self.tof_projection(tof_embedding_256))
        fused = torch.cat([fft_embedding, tof_embedding], dim=1)
        return self.head(fused)

In [151]:
# Build tensors for fusion model
X_train_fft_imu_thm_tensor = torch.from_numpy(X_train_fft_imu_thm).float()
X_val_fft_imu_thm_tensor = torch.from_numpy(X_val_fft_imu_thm).float()

def build_tof_tensor(rows_df: pd.DataFrame, first_df: pd.DataFrame) -> torch.Tensor:
    """
    Build TOF tensor aligned with first_df.

    Args:
        rows_df: DataFrame with all rows.
        first_df: DataFrame with first rows.

    Returns:
        TOF tensor.
    """
    sequence_ids = first_df[SEQ_COL].to_numpy()
    tof_arrays = []
    for seq_id in sequence_ids:
        sequence_data = rows_df[rows_df[SEQ_COL] == seq_id]
        tof_features = pad_or_truncate_time(tof_to_5x8x8(sequence_data), L_TORCH)
        tof_arrays.append(tof_features)
    return torch.from_numpy(np.stack(tof_arrays, axis=0)).float()

X_train_tof_tensor = build_tof_tensor(train_rows, train_first)
X_val_tof_tensor = build_tof_tensor(val_rows, val_first)

y_train_tensor = torch.tensor(y_train_gesture_idx, dtype=torch.long)
y_val_tensor   = torch.tensor(y_val_gesture_idx, dtype=torch.long)
N = len(y_train_gesture_idx)



In [152]:
# Train fusion model
print(" Training Intermediate Fusion Model...")
fusion_model = IntermediateFusionModel(fft_input_dim=X_train_fft_imu_thm.shape[1], num_classes=NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.AdamW(fusion_model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_function = nn.CrossEntropyLoss()

EPOCHS_FUSION = 3
BATCH_SIZE = 16
N = len(y_train_gesture_idx)

for epoch in range(1, EPOCHS_FUSION + 1):
    fusion_model.train()
    indices = np.random.permutation(N)
    for i in range(0, N, BATCH_SIZE):
        batch_indices = indices[i:i + BATCH_SIZE]
        batch_fft = X_train_fft_imu_thm_tensor[batch_indices].to(DEVICE)
        batch_tof = X_train_tof_tensor[batch_indices].to(DEVICE)
        batch_labels = y_train_tensor[batch_indices].to(DEVICE)

        optimizer.zero_grad()
        logits = fusion_model(batch_fft, batch_tof)
        loss = loss_function(logits, batch_labels)
        loss.backward()
        optimizer.step()

    # Validation sanity check
    fusion_model.eval()
    with torch.no_grad():
        val_logits = fusion_model(
            X_val_fft_imu_thm_tensor[:BATCH_SIZE].to(DEVICE),
            X_val_tof_tensor[:BATCH_SIZE].to(DEVICE)
        )
        print(f"📊 Fusion Epoch {epoch}: Val batch logits shape = {tuple(val_logits.shape)}")
    fusion_model.train()


 Training Intermediate Fusion Model...
📊 Fusion Epoch 1: Val batch logits shape = (16, 18)
📊 Fusion Epoch 2: Val batch logits shape = (16, 18)
📊 Fusion Epoch 3: Val batch logits shape = (16, 18)


In [153]:
# Save fusion model and logits
torch.save(
    {
        "state_dict": fusion_model.state_dict(),
        "num_classes": NUM_CLASSES,
        "classes": classes,
        "L_TORCH": L_TORCH,
        "L_fft": L_fft,
        "K_fft": K_fft,
        "seed": RANDOM_SEED
    },
    MODELS_DIR / "intermediate_fusion.pt"
)

fusion_model.eval()
with torch.no_grad():
    val_logits_fusion = fusion_model(X_val_fft_imu_thm_tensor.to(DEVICE), X_val_tof_tensor.to(DEVICE)).cpu().numpy()
np.save(OUTPUTS_DIR / "logits_val_intermediate_fusion.npy", val_logits_fusion)


In [154]:
# Final Sanity Check
expected_output_files = [
    "val_seq_ids.npy", "y_val_macro.npy", "y_val_bin.npy",
    "logits_val_fft_mlp_all.npy",
    "logits_val_fft_mlp_imu_thm.npy",
    "logits_val_fft_rf_all.npy",
    "logits_val_cnn_bilstm_tof.npy",
    "logits_val_late_fusion.npy",
    "logits_val_intermediate_fusion.npy",
]

missing_files = [file for file in expected_output_files if not (OUTPUTS_DIR / file).exists()]
if missing_files:
    print(f" Missing output files: {missing_files}")
else:
    print(f" All model artifacts saved to: {OUT_DIR.resolve()}")

 All model artifacts saved to: /Users/ujjwaldahiya/Desktop/Capstone-new/2026-winter-capstone-project-2026winter-capstone-group-5/models_artifacts


In [155]:
# Checking the results for the check.
# Artifact outputs
from pathlib import Path

OUT_DIR = Path("../models_artifacts")  # or ../models_artifacts / ../models_artifacts depending on your folder
MODELS_DIR = OUT_DIR / "models"
OUTPUTS_DIR = OUT_DIR / "outputs"
META_DIR = OUT_DIR / "metadata"

required_models = [
    "fft_mlp_all.joblib",
    "fft_mlp_imu_thm.joblib",
    "fft_rf_all.joblib",
    "cnn_bilstm_tof.pt",
    "intermediate_fusion.pt",
]

required_outputs = [
    "val_seq_ids.npy",
    "y_val_macro.npy",
    "y_val_bin.npy",
    "logits_val_fft_mlp_all.npy",
    "logits_val_fft_mlp_imu_thm.npy",
    "logits_val_fft_rf_all.npy",
    "logits_val_cnn_bilstm_tof.npy",
    "logits_val_late_fusion.npy",
    "logits_val_intermediate_fusion.npy",
]

required_meta = [
    "feature_cols.json",
    "split_sequence_ids.json",
    "class_mapping.json",
    "fft_config.json",
]

missing = []
for f in required_models:
    if not (MODELS_DIR / f).exists(): missing.append(f"models/{f}")
for f in required_outputs:
    if not (OUTPUTS_DIR / f).exists(): missing.append(f"outputs/{f}")
for f in required_meta:
    if not (META_DIR / f).exists(): missing.append(f"metadata/{f}")

print("Missing:", missing)
assert len(missing) == 0
print("OK: all modeling artifacts exist.")



Missing: []
OK: all modeling artifacts exist.


In [156]:
# Kernel reproducibility proof (artifact requirement)

import sys, platform
print("python:", sys.version)
print("executable:", sys.executable)
print("platform:", platform.platform())

# This must point to your project venv/bin/python. Put this info in README/report.

python: 3.14.0 (v3.14.0:ebf955df7a8, Oct  7 2025, 08:20:14) [Clang 16.0.0 (clang-1600.0.26.6)]
executable: /Users/ujjwaldahiya/CAP-@-EXP/2026-winter-capstone-project-2026winter-capstone-group-5/venv/bin/python
platform: macOS-26.2-arm64-arm-64bit-Mach-O


In [157]:
# Paper model-set compliance (all six models are represented)
import json, numpy as np
from pathlib import Path

OUT_DIR = Path("../models_artifacts")
OUTPUTS_DIR = OUT_DIR / "outputs"
META_DIR = OUT_DIR / "metadata"

with open(META_DIR / "class_mapping.json") as f:
    cm = json.load(f)

C = len(cm["classes"])
print("NUM_CLASSES:", C)

logit_files = [
    "logits_val_fft_mlp_all.npy",
    "logits_val_fft_mlp_imu_thm.npy",
    "logits_val_fft_rf_all.npy",
    "logits_val_cnn_bilstm_tof.npy",
    "logits_val_late_fusion.npy",
    "logits_val_intermediate_fusion.npy",
]

for lf in logit_files:
    arr = np.load(OUTPUTS_DIR / lf)
    print(lf, arr.shape)
    assert arr.shape[1] == C
print("OK: all logits have correct class dimension.")


NUM_CLASSES: 18
logits_val_fft_mlp_all.npy (1518, 18)
logits_val_fft_mlp_imu_thm.npy (1518, 18)
logits_val_fft_rf_all.npy (1518, 18)
logits_val_cnn_bilstm_tof.npy (1518, 18)
logits_val_late_fusion.npy (1518, 18)
logits_val_intermediate_fusion.npy (1518, 18)
OK: all logits have correct class dimension.


In [158]:
# TOF input uses exactly 320 features reshaped to (5,8,8)
import json

DATA_DIR = Path("../data/processed/cmi_sensor_data")
with open(DATA_DIR / "feature_cols.json") as f:
    feats = json.load(f)["feature_cols"]

tof_cols = [c for c in feats if c.startswith("tof_")]
print("TOF cols:", len(tof_cols))
assert len(tof_cols) == 320
print("OK: TOF feature count correct.")


TOF cols: 320
OK: TOF feature count correct.


In [159]:
# IMU+THM uses only acc/rot/thm
imu_thm_cols = [c for c in feats if c.startswith(("acc_", "rot_", "thm_"))]
bad = [c for c in imu_thm_cols if c.startswith("tof_")]
assert len(bad) == 0
print("OK: IMU+THM excludes TOF.")


OK: IMU+THM excludes TOF.


In [160]:
# Late fusion weights are correct (0.3/0.7)
import numpy as np

mlp = np.load(OUTPUTS_DIR / "logits_val_fft_mlp_imu_thm.npy")
tof = np.load(OUTPUTS_DIR / "logits_val_cnn_bilstm_tof.npy")
late = np.load(OUTPUTS_DIR / "logits_val_late_fusion.npy")

recomputed = 0.3 * mlp + 0.7 * tof
print("max abs diff:", np.max(np.abs(late - recomputed)))
assert np.max(np.abs(late - recomputed)) < 1e-6
print("OK: late fusion weights correct.")


max abs diff: 0.0
OK: late fusion weights correct.
